---
## 14. Real-World Practice — The Titanic Dataset

We've been building toward this all lecture: real historical data, 891 passenger records from
the RMS Titanic (1912) — the original, full passenger list, with a `Survived` column (0 = No,
1 = Yes) at the center of it. Every stage we practiced today — Read, Explore, Select, Filter,
Clean, Group, Combine, Transform — applies directly here.

**This section is yours.** The setup below loads the data — from here on, work through the task
list independently, the same way you'll be expected to work on real data going forward. We're
**not** doing full exploratory analysis yet (that's next lecture) — just applying today's
mechanics.

We'll finish with a short class discussion, so keep notes on anything that surprises you.

In [1]:
import pandas as pd
titanic = pd.read_csv("titanic.csv")
titanic.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


A quick note on this version of the dataset: unlike a lot of trimmed-down Titanic samples
you'll see online, this one has the full original columns, including a genuine unique key —
`PassengerId` — plus `Name`, `Ticket`, and `Cabin` in their raw form. A few things that changes
for us:
- **Duplicates become a much more meaningful check.** With a real unique ID per row, any
  `duplicated()` result (checked across all columns) tells you something is *actually* wrong with
  the data — it's not just "these passengers happen to share every visible feature" the way it
  would be without an ID.
- **`Cabin` is mostly missing** (most passengers, especially in lower classes, have no recorded
  cabin) — a different, more extreme missing-data situation than anything in our Employees table.
- **`Name` is real, raw text** — including each passenger's title (`Mr.`, `Mrs.`, `Miss.`,
  `Master.`, and a few rarer ones) embedded right in the string. That's genuinely useful data,
  not just a label, once you pull it apart with `.str` methods.

### Your tasks

1. **Explore**: run `.shape`, `.info()`, and `.describe()`. How many passengers? Which columns
   have missing data, and how much for each? (`Age`, `Cabin`, and `Embarked` are the ones to
   watch — how do their missing-value situations differ from each other?)
2. **Select**: select just `Survived`, `Pclass`, `Sex`, `Age`, and `Fare` into a new DataFrame.
3. **Filter**: find all passengers who were female, in 1st class (`Pclass == 1`), and survived
   (`Survived == 1`).
4. **Sort**: sort passengers by `Fare`, highest first, and look at the top 10 — now that we have
   real `Name` values, who were they?
5. **Create a column**: create `AgeGroup`, bucketing `Age` into `"Child"` (< 13), `"Teen"`
   (13–19), `"Adult"` (20–59), `"Senior"` (60+) — handle missing ages sensibly (decide how, and
   note your choice).
6. **Handle missing values**: check `.isnull().sum()`. Decide and justify a strategy for `Age`,
   for `Cabin`, and for `Embarked` — all three are missing for very different reasons and at very
   different rates, so a single one-size-fits-all strategy probably isn't right here.
7. **Remove duplicates**: check `titanic.duplicated().sum()`. Given the note above about
   `PassengerId` being a real unique key, what do you expect this number to be — and what would a
   *nonzero* result actually mean if you saw one?
8. **Group and aggregate**: find the survival rate (`mean` of `Survived`) grouped by `Sex`, and
   separately by `Pclass`. What pattern do you see?
9. **Pivot table**: build a pivot table of survival rate with `Pclass` as rows and `Sex` as
   columns.
10. **Transform**: extract each passenger's **title** (`Mr.`, `Mrs.`, `Miss.`, `Master.`, ...)
    out of the `Name` column using `.str` methods (hint: every title sits between a comma and a
    period — `.str.split(",")` and `.str.split(".")`, or `.str.extract()` with a regex, both get
    you there). Once you have it, check the survival rate per title — does it tell a different
    story than `Sex` alone?

Work through these in order — each builds naturally on the last, the same way today's lecture
did.

In [4]:
# Your work starts here.
# Task 1: Explore
print("titanic.info...")
titanic.info()
print("______________________________________")

print("titanic.describe...")
titanic.describe()

print("______________________________________")
print("titanic.isnull...")
titanic.isnull().sum()




titanic.info...
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB
______________________________________
titanic.describe...
______________________________________
titanic.isnull...


PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [5]:
# Task 2: Select
selected_df = titanic[['Survived', 'Pclass', 'Sex', 'Age', 'Fare']]

selected_df.head()




,Survived,Pclass,Sex,Age,Fare
0,0,3,male,22.0,7.2500
1,1,1,female,38.0,71.2833
2,1,3,female,26.0,7.9250
3,1,1,female,35.0,53.1000
4,0,3,male,35.0,8.0500


In [6]:
# Task 3: Filter

female_first_survived = titanic[(titanic['Sex'] == 'female') & (titanic['Pclass'] == 1) & (titanic['Survived'] == 1)]

female_first_survived



,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
11,12,1,1,"Bonnell, Miss. Elizabeth",female,58.0,0,0,113783,26.5500,C103,S
31,32,1,1,"Spencer, Mrs. William Augustus (Marie Eugenie)",female,NaN,1,0,PC 17569,146.5208,B78,C
52,53,1,1,"Harper, Mrs. Henry Sleeper (Myna Haxtun)",female,49.0,1,0,PC 17572,76.7292,D33,C
...,...,...,...,...,...,...,...,...,...,...,...,...
856,857,1,1,"Wick, Mrs. George Dennick (Mary Hitchcock)",female,45.0,1,1,36928,164.8667,NaN,S
862,863,1,1,"Swift, Mrs. Frederick Joel (Margaret Welles Ba...",female,48.0,0,0,17466,25.9292,D17,S
871,872,1,1,"Beckwith, Mrs. Richard Leonard (Sallie Monypeny)",female,47.0,1,1,11751,52.5542,D35,S
879,880,1,1,"Potter, Mrs. Thomas Jr (Lily Alexenia Wilson)",female,56.0,0,1,11767,83.1583,C50,C


In [7]:
# Task 4: Sort

titanic.sort_values(by='Fare', ascending=False).head(10)




,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
679,680,1,1,"Cardeza, Mr. Thomas Drake Martinez",male,36.0,0,1,PC 17755,512.3292,B51 B53 B55,C
258,259,1,1,"Ward, Miss. Anna",female,35.0,0,0,PC 17755,512.3292,NaN,C
737,738,1,1,"Lesurer, Mr. Gustave J",male,35.0,0,0,PC 17755,512.3292,B101,C
88,89,1,1,"Fortune, Miss. Mabel Helen",female,23.0,3,2,19950,263.0000,C23 C25 C27,S
438,439,0,1,"Fortune, Mr. Mark",male,64.0,1,4,19950,263.0000,C23 C25 C27,S
341,342,1,1,"Fortune, Miss. Alice Elizabeth",female,24.0,3,2,19950,263.0000,C23 C25 C27,S
27,28,0,1,"Fortune, Mr. Charles Alexander",male,19.0,3,2,19950,263.0000,C23 C25 C27,S
742,743,1,1,"Ryerson, Miss. Susan Parker ""Suzette""",female,21.0,2,2,PC 17608,262.3750,B57 B59 B63 B66,C
311,312,1,1,"Ryerson, Miss. Emily Borie",female,18.0,2,2,PC 17608,262.3750,B57 B59 B63 B66,C
299,300,1,1,"Baxter, Mrs. James (Helene DeLaudeniere Chaput)",female,50.0,0,1,PC 17558,247.5208,B58 B60,C


In [12]:
# Task 5: Create AgeGroup

titanic['Age'] = titanic['Age'].fillna(titanic['Age'].mean())
titanic['Age'].head()

def age_group(age):
    if age < 13:
        return 'Child'
    elif age <= 19:
        return 'Teen'
    elif age <= 59:
        return 'Adult'
    else:
        return 'Senior'

titanic['AgeGroup'] = titanic['Age'].apply(age_group)

titanic[['Age', 'AgeGroup']].head(10)





,Age,AgeGroup
0,22.000000,Adult
1,38.000000,Adult
2,26.000000,Adult
3,35.000000,Adult
4,35.000000,Adult
5,29.699118,Adult
6,54.000000,Adult
7,2.000000,Child
8,27.000000,Adult
9,14.000000,Teen


In [22]:
# Task 6: Handle missing values
titanic.isnull().sum()

# Age
titanic['Age'] = titanic['Age'].fillna(titanic['Age'].mean())

# Embarked
titanic['Embarked'] = titanic['Embarked'].fillna(titanic['Embarked'].mode()[0])

# Cabin
titanic['Cabin'] = titanic['Cabin'].fillna('Unknown')




In [23]:
# Task 7: Remove duplicates
titanic.duplicated().sum()

# The data is unique



np.int64(0)

In [25]:
# Task 8: Group and aggregate

titanic.groupby('Sex')['Survived'].mean()

titanic.groupby('Pclass')['Survived'].mean()




Pclass
1    0.629630
2    0.472826
3    0.242363
Name: Survived, dtype: float64

In [26]:
# Task 9: Pivot table

titanic.pivot_table(
    values='Survived',
    index='Pclass',
    columns='Sex',
    aggfunc='mean'
)



Sex,female,male
Pclass,,
1,0.968085,0.368852
2,0.921053,0.157407
3,0.500000,0.135447


In [28]:
# Task 10: Transform -- extract Title from Name
titanic['Title'] = titanic['Name'].str.extract(r',\s*([^\.]+)\.')
titanic.groupby('Title')['Survived'].mean()



Title
Capt            0.000000
Col             0.500000
Don             0.000000
Dr              0.428571
Jonkheer        0.000000
Lady            1.000000
Major           0.500000
Master          0.575000
Miss            0.697802
Mlle            1.000000
Mme             1.000000
Mr              0.156673
Mrs             0.792000
Ms              1.000000
Rev             0.000000
Sir             1.000000
the Countess    1.000000
Name: Survived, dtype: float64

### 💬 Class discussion

Once you've worked through the tasks above, be ready to discuss:
- What was the single most surprising pattern you found in survival rates?
- Which missing-value strategy did you choose for `Age`? For `Cabin`? Why might a column that's
  77%+ missing (`Cabin`) call for a completely different approach than one that's ~20% missing
  (`Age`)?
- Did `titanic.duplicated().sum()` come out to `0`? What does having a real `PassengerId` change
  about how much you can trust that result, compared to a dataset without a unique key?
- Did extracting `Title` from `Name` reveal anything survival-related that `Sex` alone didn't
  (think about `Master.` vs `Mr.`, for instance)?
- Looking back at the full workflow — Read → Explore → Select → Filter → Clean → Group → Combine
  → Transform — which single stage do you think matters most for getting *trustworthy* results?
  Why?

---
## Wrap-up

```
CSV / Excel → Read Data → Explore → Select → Filter → Clean → Group → Combine → Transform → Ready for EDA
```

Every box in that diagram is now something you can actually do. Today you practiced, on both a
small employees dataset and a real historical one:
- Creating and inspecting `Series` and `DataFrame` objects
- Reading and writing CSV and Excel files
- Exploring structure and summary statistics
- Selecting with columns, `.loc`, and `.iloc`
- Filtering with conditions, `isin()`, and `between()`
- Sorting by values and by index
- Creating, renaming, dropping, and retyping columns
- Handling missing values and duplicates, thoughtfully rather than automatically
- `groupby` and aggregation — the split-apply-combine pattern
- Combining tables with `merge()` (and every join type) and `concat()`
- Reshaping with `pivot()` and `pivot_table()`
- Transforming data with `apply()`, `map()`, `lambda`, and vectorized string methods

**Next time:** hands-on practice continues, then a Mini Project, Homework, and a Kahoot to test
understanding. After that: full Exploratory Data Analysis (EDA) — we'll take these exact
mechanics and use them to actually *investigate* a dataset, including visualization.